# 전체 코드

## 라이브러리 설치 및 임포트

In [1]:
# 필요한 라이브러리 설치
!pip install torch torchvision torchaudio
!pip install transformers sentence-transformers
!pip install qdrant-client
!pip install langchain langchain-community langchain-openai
!pip install openai
!pip install scikit-learn
!pip install rank_bm25
!pip install tiktoken
!pip install numpy tqdm
!pip install langchain-qdrant
!pip install nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.3/309.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.64.1
    Uninstalling grpcio-1.64.1:
      Successfully

In [2]:
import os
import asyncio
import logging
import time
import datetime
from typing import List, Dict, Any, Optional, Tuple
from functools import lru_cache

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client import models
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Qdrant as LangchainQdrant
from langchain.schema import Document
from langchain.retrievers import BM25Retriever
from openai import AsyncOpenAI
import nest_asyncio
from google.colab import userdata

## 기본 설정 및 초기화

In [3]:
# 설정
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
COLLECTION_NAME = "son99_d"
MODEL_NAME = "centwon/ko-gpt-trinity-1.2B-v0.5_v2"
MAX_GPT_USAGE = 0  # GPT-4 사용 횟수 제한
SIMILARITY_THRESHOLD = 0.7  # 검색 결과의 유사도 임계값

# 초기화
nest_asyncio.apply()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
openai_api_key = userdata.get('FINAL_TEAM3')
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vector_store = LangchainQdrant(client=client, collection_name=COLLECTION_NAME, embeddings=embeddings)
st_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
openai_client = AsyncOpenAI(api_key=openai_api_key)

# 전역 변수
gpt_usage_count = 0
last_reset_date = datetime.date.today()
custom_model = None
custom_tokenizer = None

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:151: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:151: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.0.37 and will be removed in 0.3.0. An updated version of the class exists in the langchain-qdrant package and should be used instead. To use it run `pip install -U langchain-qdrant` and import as `from langchain_qdrant import Qdrant`.
  warn_deprecated(


## Metadata

In [4]:
def get_unique_metadata():
    try:
        response = client.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=None,
            limit=10000,
            with_payload=True,
            with_vectors=False
        )

        categories = set()
        diseases = set()
        intents = set()

        for point in response[0]:
            categories.add(point.payload.get("질병_카테고리", ""))
            diseases.add(point.payload.get("질병", ""))
            intents.add(point.payload.get("의도", ""))

        categories.discard("")
        diseases.discard("")
        intents.discard("")

        return list(categories), list(diseases), list(intents)
    except Exception as e:
        logging.error(f"메타데이터 가져오기 중 오류 발생: {str(e)}")
        return [], [], []

# 메타데이터 초기화
DISEASE_CATEGORIES, DISEASES, INTENTS = get_unique_metadata()


In [5]:
def extract_metadata(query, categories, diseases, intents):
    query_lower = query.lower()
    category = next((cat for cat in categories if cat.lower() in query_lower), None)
    disease = next((dis for dis in diseases if dis.lower() in query_lower), None)
    intent = next((intent for intent in intents if intent.lower() in query_lower), None)
    return category, disease, intent


## QdrantRetriever

In [6]:
class QdrantRetriever:
    def __init__(self, client: QdrantClient, collection_name: str):
        self.client = client
        self.collection_name = collection_name

    def retrieve(self, query_vector: List[float], category: Optional[str] = None, disease: Optional[str] = None, intent: Optional[str] = None, top_k: int = 20) -> List[Any]:
        filter_conditions = []
        if category:
            filter_conditions.append(models.FieldCondition(key="질병_카테고리", match=models.MatchValue(value=category)))
        if disease:
            filter_conditions.append(models.FieldCondition(key="질병", match=models.MatchValue(value=disease)))
        if intent:
            filter_conditions.append(models.FieldCondition(key="의도", match=models.MatchValue(value=intent)))

        search_params = models.SearchParams(hnsw_ef=128, exact=False)

        return self.client.search(
            collection_name=self.collection_name,
            query_vector=query_vector,
            query_filter=models.Filter(must=filter_conditions) if filter_conditions else None,
            limit=top_k,
            search_params=search_params
        )

## Qdrant 검색 결과 ->  Langchain의 Document 객체 리스트로 변환

In [7]:
def create_documents_from_qdrant_results(results) -> List[Document]:
    return [Document(
        page_content=result.payload.get('답변', 'N/A'),
        metadata={
            'id': result.id,
            '질병 카테고리': result.payload.get('질병_카테고리', 'N/A'),
            '질병': result.payload.get('질병', 'N/A'),
            '부서': result.payload.get('부서', 'N/A'),
            '의도': result.payload.get('의도', 'N/A'),
            'score': result.score
        }
    ) for result in results]

In [8]:
@lru_cache(maxsize=100)
def cached_embed(query: str) -> List[float]:
    return st_model.encode(query).tolist()

## ensemble model

In [9]:
async def ensemble_search(query: str, top_k: int = 5) -> List[Document]:
    query_vector = cached_embed(query)
    qdrant_retriever = QdrantRetriever(client=client, collection_name=COLLECTION_NAME)

    category, disease, intent = extract_metadata(query, DISEASE_CATEGORIES, DISEASES, INTENTS)
    qdrant_results = qdrant_retriever.retrieve(query_vector, category, disease, intent, top_k=20)
    documents = create_documents_from_qdrant_results(qdrant_results)

    bm25_retriever = BM25Retriever.from_documents(documents)
    bm25_results = bm25_retriever.get_relevant_documents(query)

    if not bm25_results:
        return documents[:top_k]

    combined_results = []
    for bm25_result in bm25_results:
        matching_qdrant_result = next((doc for doc in documents if doc.page_content == bm25_result.page_content), None)
        if matching_qdrant_result:
            combined_score = 0.5 * matching_qdrant_result.metadata.get('score', 0) + 0.5 * bm25_result.metadata.get('score', 1)
            matching_qdrant_result.metadata['combined_score'] = combined_score
            combined_results.append(matching_qdrant_result)

    if not combined_results:
        return documents[:top_k]

    combined_results.sort(key=lambda x: x.metadata['combined_score'], reverse=True)
    return [result for result in combined_results[:top_k] if result.page_content and result.page_content.strip()]


## GPT 모델 사용량 체크

In [10]:
def check_and_reset_gpt_usage():
    global gpt_usage_count, last_reset_date
    today = datetime.date.today()
    if today > last_reset_date:
        gpt_usage_count = 0
        last_reset_date = today


## GPT4o-turbo model

In [11]:
async def generate_gpt4_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> str:
    try:
        system_message = (
            "You are an expert medical assistant designed to support seniors during medical emergencies "
            "while traveling. Your primary role is to provide accurate and clear medical advice based on the user's "
            "symptoms or questions about diseases. When the user asks about a disease or symptom, "
            "show the related metadata, such as the disease and intent, and then provide the best advice based on "
            "the retrieved information. Please respond in Korean and provide a complete response."
            "Generating an answer, please make it within max_tokens."
        )

        user_message = f"User Query: {query}\n\nContext: {context}\n\nMetadata: {metadata}"

        response = await openai_client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ],
            max_tokens=1000,
            temperature=0.3,
            n=1,
            stop=None,
        )

        generated_response = response.choices[0].message.content.strip()
        logging.info(f"GPT-4 응답 생성 완료: 길이={len(generated_response)}")
        return generated_response

    except Exception as e:
        logging.error(f"GPT-4 응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. GPT-4 응답을 생성하는 중에 오류가 발생했습니다."


## ko-gpt-trinity-1.2B-v0.5_v2
 - 우리가 파인 튜닝한 모델 v2

In [12]:
async def generate_custom_model_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> str:
    try:
        prompt = f"""질문: {query}
컨텍스트: {context or ''}
메타데이터: {metadata or {}}

위 정보를 바탕으로 질문에 대한 구체적이고 정확한 답변을 제공해주세요. 답변은 다음 형식을 따라주세요:
1. 질문과 직접적으로 관련된 핵심 정보를 먼저 제시합니다.
2. 가능한 경우 구체적인 증상이나 치료 방법을 언급합니다.
3. 의학적 조언이 필요한 경우, 어떤 전문의를 찾아가야 하는지 명시합니다.

답변:"""

        inputs = custom_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(custom_model.device)

        with torch.no_grad():
            outputs = custom_model.generate(
                **inputs,
                max_length=300,
                num_return_sequences=1,
                no_repeat_ngram_size=3,
                temperature=0.7,
                do_sample=True,
                top_k=50,
                top_p=0.95,
            )

        response = custom_tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = response.split("답변:")[-1].strip()

        return response

    except Exception as e:
        logging.error(f"커스텀 모델 응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. 응답을 생성하는 중에 오류가 발생했습니다."


## model 응답 생성 관리

In [13]:

async def generate_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> Tuple[str, str]:
    global gpt_usage_count

    try:
        if gpt_usage_count < MAX_GPT_USAGE:
            gpt_usage_count += 1
            response = await generate_gpt4_response(query, context, metadata)
            return response, "GPT-4"
        else:
            response = await generate_custom_model_response(query, context, metadata)
            return response, "Custom Model"
    except Exception as e:
        logging.error(f"응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return "죄송합니다. 응답을 생성하는 중에 오류가 발생했습니다.", "Error"


## 사용자 쿼리 처리

In [14]:
async def process_query_async(query: str) -> Tuple[str, str, float]:
    start_time = time.time()
    logging.info(f"처리 중인 쿼리: {query}")

    try:
        ensemble_results, (initial_response, model) = await asyncio.gather(
            ensemble_search(query, 5),
            generate_response(query)
        )

        if not ensemble_results:
            logging.warning("앙상블 검색 결과가 없습니다.")
            return initial_response, model, time.time() - start_time

        best_match = ensemble_results[0]
        context = best_match.page_content
        metadata = best_match.metadata

        print(f"\nDB에서 검색된 정보:\n내용: {context}\n메타데이터: {metadata}\n")

        if metadata.get('combined_score', 0) < SIMILARITY_THRESHOLD:
            logging.warning("질문과 관련된 정보를 찾기 어렵습니다.")
            response = initial_response
        else:
            response, model = await generate_response(query, context, metadata)

        processing_time = time.time() - start_time
        logging.info(f"처리 시간: {processing_time:.2f}초")

        return response, model, processing_time

    except Exception as e:
        logging.error(f"쿼리 처리 중 오류 발생: {str(e)}", exc_info=True)
        response, model = await generate_response(query)
        return response, model, time.time() - start_time


In [15]:
async def run_system():
    print("의료 정보 시스템을 초기화하는 중...")

    try:
        global custom_model, custom_tokenizer
        client.get_collections()
        print("Qdrant 서버에 성공적으로 연결되었습니다.")

        if not openai_api_key:
            raise ValueError("OpenAI API 키가 설정되지 않았습니다.")
        print("OpenAI API 키가 확인되었습니다.")

        # 커스텀 모델 초기화 (한 번만 수행)
        if custom_model is None or custom_tokenizer is None:
            custom_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to('cuda' if torch.cuda.is_available() else 'cpu')
            custom_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
            print("커스텀 모델과 토크나이저가 초기화되었습니다.")

        print("\n의료 정보 시스템이 성공적으로 초기화되었습니다.")
        print("시스템이 준비되었습니다. 질문을 입력해주세요.")

        return True

    except Exception as e:
        print(f"시스템 초기화 중 오류가 발생했습니다: {str(e)}")
        print("시스템을 종료합니다.")
        return False

## 실행

In [16]:
async def main():
    if not await run_system():
        return

    while True:
        user_query = input("질문을 입력하세요 (종료하려면 'quit' 입력): ")
        if user_query.lower() == 'quit':
            print("프로그램을 종료합니다.")
            break

        try:
            response, model, processing_time = await process_query_async(user_query)
            print(f"\n답변 ({model}):\n{response}")
            print(f"처리 시간: {processing_time:.2f}초\n")
        except Exception as e:
            print(f"오류 발생: {str(e)}")
            print("다시 질문해 주세요.")

if __name__ == "__main__":
    asyncio.run(main())

의료 정보 시스템을 초기화하는 중...
Qdrant 서버에 성공적으로 연결되었습니다.
OpenAI API 키가 확인되었습니다.


config.json:   0%|          | 0.00/868 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/64.0k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

커스텀 모델과 토크나이저가 초기화되었습니다.

의료 정보 시스템이 성공적으로 초기화되었습니다.
시스템이 준비되었습니다. 질문을 입력해주세요.
질문을 입력하세요 (종료하려면 'quit' 입력): 고혈압 증상을 알려줘.


/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:151: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  warn_deprecated(



DB에서 검색된 정보:
내용: 고혈압 환자는 두통, 어지러움, 심박수 증가 등 다양한 증상을 느낄 수 있습니다. 이러한 증상은 고혈압이 원인으로 발생하는 것일 수 있습니다. 일부 사람들은 고혈압의 원인 질환을 인식하지 못하고 있을 수도 있으므로, 정기적인 혈압 검사를 받는 것이 중요합니다.
메타데이터: {'id': 3882, '질병 카테고리': '순환기질환', '질병': '고혈압', '부서': '내과', '의도': '증상', 'score': 0.74013335, 'combined_score': 0.74013335}


답변 (Custom Model):
정기적인 검사는 고혈압과 심혈관 건강을 지키는 데 중요합니다. 전문의의 정확한 상담과 처치가 필요합니다. 필요시 추가 검사가 필요할 수 있습니다. 정확한 검사는 혈압 조절과 건강 관리에 도움을 줄 수 있습니다. 추가적인 정보 제공은 혈압과 심전도를 평가하는 데 도움을 줄 것입니다. 필요시 의사의 추가 설명이 필요할 수 있습니다.
 치료 방법과 관련한 추가적인 정보는 의사의 상담과 평가에서 확인 가능합니다. 자세한 정보는 의료 전문가의 상담이 필요합니다. 추가 정보는 의사에게 문의하는 것이 좋습니다.
처리 시간: 6.05초

질문을 입력하세요 (종료하려면 'quit' 입력): 식중독 증상에 대해서 알려줘.



DB에서 검색된 정보:
내용: 식중독은 다양한 증상을 보이며, 증상에 따라 구토와 설사가 가장 흔하고, 복통, 근육통, 두통 등의 신체 증상도 동반될 수 있습니다. 식중독은 세균에 의해 감염되어 발생하므로, 음식물의 오염 정도가 심하지 않은 경우에는 특별히 치료가 필요하지 않을 수 있습니다. 식중독의 원인은 미생물의 침투와 음식 내부의 독소 확산에 기인합니다. 그러나 구토나 설사가 지속되거나 심해지는 경우에는 병원을 방문하여 치료를 받는 것이 좋습니다.
메타데이터: {'id': 12727, '질병 카테고리': '응급질환', '질병': '식중독', '부서': '내과', '의도': '증상', 'score': 0.6760503, 'combined_score': 0.6760503}


답변 (Custom Model):
식중독 증상은 구토, 설사, 복통, 발열, 발진 등이 있습니다. 증상은 병원 치료와 적절한 위생 관리가 중요합니다. 증상에 따라 적절한 치료를 받는 것이 중요합니다. 감염원이 확인되면 추가 검사를 고려해야 합니다. 증상이 심하거나 지속되면 병원에서 진료를 받는 것이 좋습니다. 정확한 검사가 필요합니다. 증상은 식중독의 경위와 원인을 파악하는 데 도움이 됩니다. 정확한 정보가 필요합니다. 필요 시 의료 전문가의 도움을 받는 것이 바람직합니다. 
 치료는 증상에 맞춰 진행됩니다. 치료가 필요하면 병원에서 추가 검사를 진행합니다. 치료는 항생제, 해열제, 진정제 등을 통해 이뤄집니다. 치료가 늦어지면 증상이 악화될 수 있습니다. 증상 완화가 우선입니다. 
 증상에 맞는 적절한 치료를 받을 수 있도록 지도합니다. 증상과 치료 계획을 확인하고 조처하는 것이 중요합니다.
 
 출처 : T.397. Wikipedia, 식중독 정보, 식약처, 질병관리본부
 	
 <식중독 예방법>
  철저한 손 씻기와 안전한 음식 제공이 중요합니다. 음식은 냉장 보관하고 유통기한을 확인합니다. 오염된 음식은 신속하게 제거하고
처리 시간: 5.67초

질문을 입력하세요 (종료하려면 '